In [0]:

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F


# ============================================================
# 2. PROJECT CONFIGURATION
# ============================================================

CATALOG = "workspace"

BASE_PATH = (
    "/Volumes/workspace/default/data_volume/"
    "ecommerce_lakehouse"
)

LANDING_PATH = f"{BASE_PATH}/landing"

CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/bronze"

SCHEMA_PATH = f"{BASE_PATH}/schemas/bronze"

BRONZE_SCHEMA = "ecommerce_bronze"

DATASETS = [
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "inventory_snapshots"
]

print("Project configuration completed.")


# ============================================================
# 3. CREATE BRONZE SCHEMA
# ============================================================

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    {CATALOG}.{BRONZE_SCHEMA}
    """
)

print("Bronze schema ready.")


# ============================================================
# 4. BRONZE INGESTION FUNCTION
# ============================================================

def ingest_bronze(dataset):

    source_path = f"{LANDING_PATH}/{dataset}"

    checkpoint_path = f"{CHECKPOINT_PATH}/{dataset}"

    schema_path = f"{SCHEMA_PATH}/{dataset}"

    table_name = (
        f"{CATALOG}.{BRONZE_SCHEMA}.bronze_{dataset}"
    )

    print("\n" + "=" * 70)
    print(f"BRONZE INGESTION: {dataset.upper()}")
    print("=" * 70)

    # --------------------------------------------------------
    # 4.1 VALIDATE SOURCE FILES
    # --------------------------------------------------------

    source_files = [
        file
        for file in dbutils.fs.ls(source_path)
        if file.path.lower().endswith(".csv")
    ]

    if not source_files:

        raise ValueError(
            f"No CSV files found: {source_path}"
        )

    print(f"CSV files found: {len(source_files)}")


    # --------------------------------------------------------
    # 4.2 READ USING AUTO LOADER
    # --------------------------------------------------------

    source_df = (

        spark.readStream

        .format("cloudFiles")

        .option("cloudFiles.format", "csv")

        .option(
            "cloudFiles.schemaLocation",
            schema_path
        )

        .option(
            "cloudFiles.inferColumnTypes",
            "true"
        )

        .option("header", "true")

        .option("mode", "PERMISSIVE")

        .load(source_path)

    )


    # --------------------------------------------------------
    # 4.3 ADD INGESTION METADATA
    # --------------------------------------------------------

    bronze_df = (

        source_df

        .withColumn(
            "_ingested_at",
            F.current_timestamp()
        )

        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )

    )


    # --------------------------------------------------------
    # 4.4 WRITE TO UNITY CATALOG DELTA TABLE
    # --------------------------------------------------------

    query = (

        bronze_df

        .writeStream

        .format("delta")

        .outputMode("append")

        .option(
            "checkpointLocation",
            checkpoint_path
        )

        .option(
            "mergeSchema",
            "true"
        )

        .trigger(
            availableNow=True
        )

        .toTable(table_name)

    )

    query.awaitTermination()

    print("Auto Loader ingestion completed.")


    # --------------------------------------------------------
    # 4.5 READ BRONZE TABLE
    # --------------------------------------------------------

    result_df = spark.table(table_name)

    record_count = result_df.count()

    if record_count == 0:

        raise ValueError(
            f"No records found in {table_name}"
        )


    # --------------------------------------------------------
    # 4.6 VALIDATE METADATA
    # --------------------------------------------------------

    required_columns = [
        "_ingested_at",
        "_source_file"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in result_df.columns
    ]

    if missing_columns:

        raise ValueError(
            f"Missing metadata columns: {missing_columns}"
        )


    # --------------------------------------------------------
    # 4.7 DISPLAY RESULTS
    # --------------------------------------------------------

    print("\nINGESTION SUCCESSFUL")

    print(f"Dataset      : {dataset}")
    print(f"Record Count : {record_count}")
    print(f"Table        : {table_name}")

    display(result_df.limit(5))


    return {

        "dataset": dataset,
        "records": record_count,
        "table_name": table_name,
        "status": "SUCCESS",
        "error": ""

    }


# ============================================================
# 5. PROCESS ALL DATASETS
# ============================================================

ingestion_results = []

for dataset in DATASETS:

    try:

        result = ingest_bronze(dataset)

        ingestion_results.append(result)

    except Exception as e:

        error_message = str(e)[:2000]

        print("\n" + "!" * 70)
        print(f"FAILED DATASET: {dataset}")
        print(error_message)
        print("!" * 70)

        ingestion_results.append({

            "dataset": dataset,
            "records": 0,
            "table_name": (
                f"{CATALOG}.{BRONZE_SCHEMA}.bronze_{dataset}"
            ),
            "status": "FAILED",
            "error": error_message

        })


# ============================================================
# 6. INGESTION SUMMARY
# ============================================================

summary_df = spark.createDataFrame(
    ingestion_results
)

display(summary_df.orderBy("dataset"))


# ============================================================
# 7. EXECUTION STATISTICS
# ============================================================

success_count = sum(
    1
    for row in ingestion_results
    if row["status"] == "SUCCESS"
)

failed_count = sum(
    1
    for row in ingestion_results
    if row["status"] == "FAILED"
)

total_records = sum(
    row["records"]
    for row in ingestion_results
)

print("\n" + "=" * 70)
print("BRONZE INGESTION SUMMARY")
print("=" * 70)

print(f"Total datasets : {len(DATASETS)}")
print(f"Successful     : {success_count}")
print(f"Failed         : {failed_count}")
print(f"Total records  : {total_records}")


# ============================================================
# 8. FINAL VALIDATION
# ============================================================

if failed_count > 0:

    failed_datasets = [
        row["dataset"]
        for row in ingestion_results
        if row["status"] == "FAILED"
    ]

    raise RuntimeError(
        f"Bronze ingestion failed: {failed_datasets}"
    )


print("\n" + "=" * 70)
print("ALL BRONZE DATASETS PROCESSED SUCCESSFULLY")
print("=" * 70)


In [0]:
for dataset in DATASETS:

    table_name = (
        f"{CATALOG}.{BRONZE_SCHEMA}.bronze_{dataset}"
    )

    exists = spark.catalog.tableExists(table_name)

    print(table_name, "EXISTS" if exists else "NOT FOUND")

In [0]:
# ============================================================
# 01. BRONZE INGESTION - AUTO LOADER
# E-COMMERCE LAKEHOUSE PROJECT
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from pyspark.sql import functions as F

print("Libraries imported successfully.")


# ============================================================
# 2. PROJECT CONFIGURATION
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"

BASE_PATH = (
    "/Volumes/workspace/default/data_volume/"
    "ecommerce_lakehouse"
)

LANDING_PATH = f"{BASE_PATH}/landing"

CHECKPOINT_BASE_PATH = (
    f"{BASE_PATH}/checkpoints/bronze"
)

SCHEMA_BASE_PATH = (
    f"{BASE_PATH}/schemas/bronze"
)

DATASETS = [
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "inventory_snapshots"
]

# Safety setting:
# Existing Bronze tables will never be re-ingested.
# Missing tables can be created using Auto Loader.

RUN_AUTO_LOADER_INGESTION = True

# Expected metadata columns in your Bronze tables.

REQUIRED_METADATA_COLUMNS = [
    "_rescued_data",
    "_source_file",
    "_source_file_modification_time",
    "_ingestion_timestamp",
    "_ingestion_date"
]

print("Project configuration completed.")


# ============================================================
# 3. CREATE BRONZE SCHEMA
# ============================================================

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{BRONZE_SCHEMA}"
)

print("Bronze schema is ready.")


# ============================================================
# 4. AUTO LOADER INGESTION FUNCTION
# ============================================================

def ingest_dataset(dataset):

    print("\n" + "=" * 75)
    print(f"BRONZE INGESTION: {dataset.upper()}")
    print("=" * 75)

    source_path = (
        f"{LANDING_PATH}/{dataset}"
    )

    table_name = (
        f"{CATALOG}.{BRONZE_SCHEMA}.bronze_{dataset}"
    )

    checkpoint_path = (
        f"{CHECKPOINT_BASE_PATH}/{dataset}_v2"
    )

    schema_path = (
        f"{SCHEMA_BASE_PATH}/{dataset}_v2"
    )

    # --------------------------------------------------------
    # 4.1 CHECK SOURCE DIRECTORY
    # --------------------------------------------------------

    try:

        files = dbutils.fs.ls(source_path)

        csv_files = [
            file for file in files
            if file.name.lower().endswith(".csv")
        ]

        print("Source directory:", source_path)
        print("CSV files found:", len(csv_files))

        if len(csv_files) == 0:

            print("No CSV files found. Skipping ingestion.")

            return {
                "dataset": dataset,
                "status": "SKIPPED",
                "records": 0
            }

    except Exception as e:

        print("Source directory error:", str(e)[:500])

        return {
            "dataset": dataset,
            "status": "FAILED",
            "records": 0
        }


    # --------------------------------------------------------
    # 4.2 CHECK EXISTING BRONZE TABLE
    # --------------------------------------------------------

    try:

        table_exists = spark.catalog.tableExists(
            table_name
        )

        if table_exists:

            existing_df = spark.table(table_name)

            existing_count = existing_df.count()

            print("Bronze table already exists.")
            print("Existing records:", existing_count)

            print(
                "Skipping ingestion to prevent "
                "duplicate records."
            )

            return {
                "dataset": dataset,
                "status": "EXISTING",
                "records": existing_count
            }

    except Exception as e:

        print("Table check failed:", str(e)[:500])

        return {
            "dataset": dataset,
            "status": "FAILED",
            "records": 0
        }


    # --------------------------------------------------------
    # 4.3 INGESTION SAFETY CHECK
    # --------------------------------------------------------

    if not RUN_AUTO_LOADER_INGESTION:

        print("Auto Loader ingestion is disabled.")

        return {
            "dataset": dataset,
            "status": "SKIPPED",
            "records": 0
        }


    # --------------------------------------------------------
    # 4.4 AUTO LOADER SOURCE
    # --------------------------------------------------------

    try:

        print("Starting Auto Loader...")

        source_df = (
            spark.readStream
            .format("cloudFiles")

            .option(
                "cloudFiles.format",
                "csv"
            )

            .option(
                "cloudFiles.schemaLocation",
                schema_path
            )

            .option(
                "cloudFiles.inferColumnTypes",
                "true"
            )

            .option(
                "cloudFiles.schemaEvolutionMode",
                "rescue"
            )

            .option(
                "cloudFiles.rescuedDataColumn",
                "_rescued_data"
            )

            .option(
                "header",
                "true"
            )

            .option(
                "mode",
                "PERMISSIVE"
            )

            .load(source_path)
        )


        # ----------------------------------------------------
        # 4.5 ADD INGESTION METADATA
        # ----------------------------------------------------

        bronze_df = (
            source_df

            .withColumn(
                "_source_file",
                F.col("_metadata.file_path")
            )

            .withColumn(
                "_source_file_modification_time",
                F.col("_metadata.file_modification_time")
            )

            .withColumn(
                "_ingestion_timestamp",
                F.current_timestamp()
            )

            .withColumn(
                "_ingestion_date",
                F.current_date()
            )

            .drop("_metadata")
        )


        # ----------------------------------------------------
        # 4.6 WRITE TO MANAGED BRONZE TABLE
        # ----------------------------------------------------

        print("Writing data to Bronze table...")

        query = (
            bronze_df.writeStream

            .format("delta")

            .outputMode("append")

            .option(
                "checkpointLocation",
                checkpoint_path
            )

            .option(
                "mergeSchema",
                "true"
            )

            .trigger(
                availableNow=True
            )

            .toTable(table_name)
        )

        query.awaitTermination()

        print("Auto Loader ingestion completed.")


        # ----------------------------------------------------
        # 4.7 VERIFY INGESTED TABLE
        # ----------------------------------------------------

        result_df = spark.table(table_name)

        record_count = result_df.count()

        print("Bronze table:", table_name)
        print("Total records:", record_count)

        return {
            "dataset": dataset,
            "status": "SUCCESS",
            "records": record_count
        }


    except Exception as e:

        print(
            "INGESTION ERROR:",
            str(e)[:1500]
        )

        return {
            "dataset": dataset,
            "status": "FAILED",
            "records": 0
        }


# ============================================================
# 5. EXECUTE AUTO LOADER FOR ALL DATASETS
# ============================================================

ingestion_results = []

for dataset in DATASETS:

    result = ingest_dataset(dataset)

    ingestion_results.append(result)


# ============================================================
# 6. INGESTION SUMMARY
# ============================================================

print("\n")
print("=" * 75)
print("BRONZE INGESTION SUMMARY")
print("=" * 75)

total_records = 0

for result in ingestion_results:

    print(
        f"{result['dataset']:<25}"
        f"{result['status']:<15}"
        f"{result['records']}"
    )

    total_records += result["records"]


success_count = sum(
    1 for result in ingestion_results
    if result["status"] in ["SUCCESS", "EXISTING"]
)

failed_count = sum(
    1 for result in ingestion_results
    if result["status"] == "FAILED"
)

print("-" * 75)

print("Total datasets:", len(DATASETS))
print("Successful or existing:", success_count)
print("Failed datasets:", failed_count)
print("Total Bronze records:", total_records)

print("=" * 75)


# ============================================================
# 7. FINAL BRONZE TABLE VALIDATION
# ============================================================

print("\n")
print("=" * 75)
print("BRONZE TABLE VALIDATION")
print("=" * 75)

validation_results = []

for dataset in DATASETS:

    table_name = (
        f"{CATALOG}.{BRONZE_SCHEMA}.bronze_{dataset}"
    )

    try:

        if not spark.catalog.tableExists(table_name):

            print(f"{dataset}: TABLE NOT FOUND")

            validation_results.append({
                "dataset": dataset,
                "records": 0,
                "missing_columns": REQUIRED_METADATA_COLUMNS,
                "status": "FAILED"
            })

            continue


        df = spark.table(table_name)

        record_count = df.count()

        missing_columns = [
            column
            for column in REQUIRED_METADATA_COLUMNS
            if column not in df.columns
        ]

        null_source_count = (
            df.filter(
                F.col("_source_file").isNull()
            ).count()
            if "_source_file" in df.columns
            else -1
        )

        null_timestamp_count = (
            df.filter(
                F.col("_ingestion_timestamp").isNull()
            ).count()
            if "_ingestion_timestamp" in df.columns
            else -1
        )

        if (
            record_count > 0
            and len(missing_columns) == 0
            and null_source_count == 0
            and null_timestamp_count == 0
        ):

            status = "PASSED"

        else:

            status = "FAILED"


        validation_results.append({
            "dataset": dataset,
            "records": record_count,
            "missing_columns": missing_columns,
            "status": status
        })


        print(
            f"{dataset:<25}"
            f"Records: {record_count:<10}"
            f"Status: {status}"
        )

        if missing_columns:

            print(
                "Missing metadata columns:",
                missing_columns
            )

        if null_source_count > 0:

            print(
                "Null source file records:",
                null_source_count
            )

        if null_timestamp_count > 0:

            print(
                "Null ingestion timestamp records:",
                null_timestamp_count
            )


    except Exception as e:

        print(
            f"{dataset}: VALIDATION ERROR - {str(e)[:500]}"
        )

        validation_results.append({
            "dataset": dataset,
            "records": 0,
            "missing_columns": [],
            "status": "FAILED"
        })


# ============================================================
# 8. FINAL VALIDATION SUMMARY
# ============================================================

passed_count = sum(
    1 for result in validation_results
    if result["status"] == "PASSED"
)

validation_failed_count = sum(
    1 for result in validation_results
    if result["status"] == "FAILED"
)

print("\n")
print("=" * 75)
print("FINAL BRONZE LAYER SUMMARY")
print("=" * 75)

print("Total datasets:", len(DATASETS))
print("Validated successfully:", passed_count)
print("Validation failures:", validation_failed_count)

print(
    "Total Bronze records:",
    sum(
        result["records"]
        for result in validation_results
    )
)

print("=" * 75)


# ============================================================
# 9. FINAL EXECUTION STATUS
# ============================================================

if (
    failed_count > 0
    or validation_failed_count > 0
):

    raise Exception(
        "BRONZE INGESTION OR VALIDATION FAILED. "
        "Check the errors above."
    )

else:

    print("\n")
    print("=" * 75)
    print("BRONZE LAYER EXECUTION COMPLETED SUCCESSFULLY")
    print("=" * 75)